In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

TESTBED = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(TESTBED))

import numpy as np
from pyfluids import FluidsList, Input

from otp.propellants.propellant import Propellant
from otp.engine.records import PropCEA, EngineRequirements, EngineChoices, InjectorChoices, EngineCalibration
from otp.engine.sizing import size_engine
from otp.engine.analysis import Engine
from otp.pump.records import PumpRequirements, PumpChoices
from otp.pump.sizing import size_pump
from otp.pump.analysis import Pump
from otp.plotting.panels import Panel, sweep_plot
from otp.core.units import g, K_TO_C
from otp.turbine.records import TurbineRequirements

import copy as cp

lox = Propellant("LOX", FluidsList.Oxygen, t_tank=89, p_tank=3)
rp1 = Propellant("RP-1", FluidsList.nDodecane, t_tank=293.15, p_tank=3)

propcea = PropCEA(lox, rp1)
engine_req = EngineRequirements(F=12000, p_a_bar=1.01325)
engine_choices = EngineChoices(OF=2, p_c_bar=50, p_e_bar=1, perc_film=0.1)
injector_choices = InjectorChoices(inj_dp=10, fuel_cd=0.8, ox_cd=0.8)
engine_calib = EngineCalibration(eff_cstar=0.95)
chamber, injector = size_engine(engine_req, engine_choices, injector_choices, engine_calib, propcea)
engine = Engine(chamber, injector, propcea, engine_calib)
engine_perf = engine.point_performance(p_c_bar=50, p_a_bar=1.01325, OF=2)
ox_pump_req = PumpRequirements.from_engine_perf(engine_perf, "ox")
ox_pump_choices = PumpChoices(rpm=30000, n_blades=6, v_inlet=11) # V_inlet hugely changes things here
ox_pump_geometry = size_pump(ox_pump_req, ox_pump_choices, sizing_method="lock")
ox_pump = Pump(ox_pump_req, ox_pump_choices, ox_pump_geometry)
ox_pump_perf = ox_pump.point_performance(propellant=lox, Q=ox_pump_req.Q_req, rpm=ox_pump_choices.rpm, p_upstream=lox.p_tank, T_upstream=lox.t_tank, method="lock")
turbine_req = TurbineRequirements.from_pump_perfs([ox_pump_perf])

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
18299.042452118556
36598.08490423711


In [ ]:
# _ok = lambda p: p.H_total_real > 0          # hide post-cutoff garbage
_ok = lambda p: True
PANELS = [
    Panel("Pressure Rise (bar)", "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.dp),
    Panel("Power Efficiency",    "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.eta_power),
    Panel("Head Coefficient",    "Flow coefficient", lambda p: p.flow_coeff_outlet, lambda p: p.head_coeff),
    Panel("Specific Speed",      "Flow rate (L/s)", lambda p: p.Q*1000,
          lambda p: p.rpm*np.sqrt(p.Q)/p.H_total_real**0.75 if p.H_total_real > 0 else np.nan),
    Panel("NPSH throat (m)",     "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.npsh_r_throat,
          ref=lambda p: p.npsh_a_upstream, mask=_ok, secondary=("bar", lambda x: x * lox.tank_fluid.density * g / 1e5, lambda x: x / (lox.tank_fluid.density * g / 1e5))),
    Panel("NPSH inlet (m)",      "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.npsh_r_inlet,
          ref=lambda p: p.npsh_a_upstream, mask=_ok, secondary=("bar", lambda x: x * lox.tank_fluid.density * g / 1e5, lambda x: x / (lox.tank_fluid.density * g / 1e5))),
    Panel("Inlet pressure (bar)",          "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.p_inlet, ref=lambda p: lox.fluid.with_state(Input.quality(0), Input.temperature(lox.t_tank + K_TO_C)).pressure / 1e5),
    Panel("Impeller inlet pressure (bar)", "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.p_1, ref=lambda p: lox.fluid.with_state(Input.quality(0), Input.temperature(lox.t_tank + K_TO_C)).pressure / 1e5),
    Panel("Diffuser throat pressure (bar)","Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.p_throat, mask=_ok, ref=lambda p: lox.fluid.with_state(Input.quality(0), Input.temperature(lox.t_tank + K_TO_C)).pressure / 1e5),
    Panel("AI NPSHR Low", "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.npsh_r_ai_low, ref=lambda p: p.npsh_a_upstream, secondary=("bar", lambda x: x * lox.tank_fluid.density * g / 1e5, lambda x: x / (lox.tank_fluid.density * g / 1e5))),
    Panel("AI NPSHR High", "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.npsh_r_ai_high, ref=lambda p: p.npsh_a_upstream, secondary=("bar", lambda x: x * lox.tank_fluid.density * g / 1e5, lambda x: x / (lox.tank_fluid.density * g / 1e5))),
    Panel("Impeller Inlet Meridian Velocity", "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.w_1),
    Panel("Impeller Inlet Circumferential Velocity", "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.u_1),
]

rpm_sweep = sweep_plot(ox_pump, PANELS, Qs=np.linspace(0, 8e-3, 20),
           sweep=("rpm", [10000, 15000, 20000, 25000, 30000]),
           fixed=dict(propellant=lox, p_upstream=lox.p_tank, T_upstream=lox.t_tank),
           methods=["lock", "barske"], design=ox_pump_perf)

p_upstream_sweep = sweep_plot(ox_pump, PANELS, Qs=np.linspace(0, 8e-3, 20),
           sweep=("p_upstream", [1, 2, 3, 4, 5]),
           fixed=dict(propellant=lox, rpm=30000, T_upstream=lox.t_tank),
           methods=["lock", "barske"], design=ox_pump_perf)